# 科技板块恐慌指数 PI · 快速上手（Google Colab）

**不用在本机安装任何东西。** 依次运行下面的单元格即可。

预计耗时 3~8 分钟，绝大部分时间花在下载 QVIX 期权数据上（上游限速约 8 KB/s，正常现象）。

> ⚠️ 两个口径（A 完整 / B 即时）**不可相减**，原因见最后一段。
> 本指标为研究/观测级度量，不预测涨跌方向，不构成投资建议。

## 1. 获取代码

In [ ]:
# 如果没有自己的 fork，请把地址换成你的仓库
!git clone -q https://github.com/jymadrid/techpanic.git
%cd techpanic

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## 2. 环境自检

In [ ]:
!python -m techpanic --check --no-color

## 3. 计算今日读数

这一步抓取两类公开数据：

1. 指数日线（新浪财经，约 1 秒）
2. QVIX 期权隐含波动率（1.optbbs.com，914 KB，**1.7 秒 ~ 123 秒不等**）

超时会自动降级用缓存并明确标注 —— 那不是报错。

退出码：0=成功，2=部分降级（正常），3=无数据。

In [ ]:
!python -m techpanic --no-color

## 4. 用 Python 读取结果

In [ ]:
import json, pandas as pd

latest = json.load(open('data/output/latest.json', encoding='utf-8'))
for r in latest['results']:
    a, b = r['full'], r['price_only']
    print(r['name'], ' 最新交易日', r['latest_trading_day'])
    ta = '无值' if not a['available'] else '%.1f %s (数据日 %s)' % (a['value'], a['level'], a['data_date'])
    tb = '无值' if not b['available'] else '%.1f %s (数据日 %s)' % (b['value'], b['level'], b['data_date'])
    print('  A 完整口径', ta)
    print('  B 即时口径', tb)
    print('  方向', r['direction'], ' 成分 S=%s A=%s F=%s' % (r['components']['S'], r['components']['A'], r['components']['F']))
    print()

In [ ]:
df = pd.read_csv('data/output/panic_index_tech_kcb.csv', encoding='utf-8-sig')
print(len(df), '行，从', df['date'].iloc[0], '到', df['date'].iloc[-1])
df.tail(10)

## 5. 画一张图

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
tail = df.tail(500).copy()
ax.plot(tail['date'], tail['PI完整口径'], label='A 完整口径（含期权）', lw=1.2)
ax.plot(tail['date'], tail['PI即时口径'], label='B 即时口径（仅价格）', lw=1.2, alpha=0.85)
ax.axhline(75, color='orange', ls=':', lw=1, label='p75 = 恐慌线（历史）')
ax.set_title('科技板块恐慌指数 PI（近 500 个交易日）')
ax.set_ylabel('PI (0-100)')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. 怎么读这两个数

| | A 完整口径 | B 即时口径 |
|---|---|---|
| 因子 | 波动抬升 + 下跌不对称 + **期权隐含波动** | 只有价格两个因子 |
| 数据日 | QVIX 盘后发布，**通常滞后 1 个交易日** | 当日收盘 |
| 用途 | 研究、引用、官方读数 | 看今天到底怎么了 |

**关键：两者不要相减。** 差值里混着「口径不同」和「数据日不同」两个原因，无法拆开。
想看今天的新增冲击，比较**同一个口径**的今天与昨天。

**判读：** PI 高 + 方向向下 = 恐慌；PI 高 + 方向向上 = 狂热（不是恐慌）。

PI 是温度计，不是预言机：它不预测涨跌，不构成投资建议。详见仓库 DISCLAIMER.md。